# tool-call-agent: fine-tuning a small LLM to pick and fill tool calls

I built this during an internship at a telecom company. The question: can a small open model, fine-tuned on a free GPU, read a plain request like "Top up line 9 with 50 bucks", decide which backend tool it needs, and write JSON a Java service can execute?

```json
{"tool": "chargeLine", "parameters": {"lineId": "9", "amount": 50.0}}
```

This is version 2. In version 1 the tool name was written into the prompt before the model started, so the model only filled in the line id and amount, and it was trained and tested on `chargeLine` alone. That version is kept under the git tag `v1-single-tool`. Here the model has to choose between three tools, or decline the request with a reason.

### Contents

1. Install dependencies and check the GPU
2. Load Gemma 4 E2B (4-bit) and attach LoRA adapters
3. Generate a synthetic dataset with three tools and a "none" class
4. Tokenize and fine-tune
5. Run inference, extract the JSON and validate it
6. Evaluate on 60 hand-written test prompts

### Before you run it

- Use a T4 GPU runtime or better (in Colab: Runtime > Change runtime type).
- Run the install cell, restart the session, then run everything from the top of section 2.
- All data is synthetic and generated below. Nothing here comes from real customers or company systems.

### Known limits

- The 240 training examples come from a fixed set of templates, so the phrasing is narrow.
- The test set is 60 prompts I wrote by hand. That is enough to spot clear failure patterns, and too small for precise accuracy numbers.
- The thought line is short and templated. It records the decision, and there is no real reasoning in it.
- There is no constrained decoding, so the model can still write malformed JSON. `parse_action` rejects it.
- I haven't run this version yet, so the notebook has no results.

## 1. Setup

### Install dependencies

Unsloth patches the model for faster, lower-memory fine-tuning. The `trl` pin is the version that worked for this run. An earlier version of this cell also pinned `xformers<0.0.30`, but that failed to build on current Colab, and Unsloth runs without xformers, so the pin was dropped.

After this cell finishes, restart the session (Runtime > Restart session, or Ctrl+M .) so Python picks up the new packages.

> Note: on my original run the `unsloth_zoo` import failed and I patched a few missing functions into the package by hand. I removed that hack from this version. If you hit `ImportError`s from `unsloth_zoo` on a newer or older release, reinstalling both `unsloth` and `unsloth-zoo` from GitHub with `--no-cache-dir` was what fixed it for me.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.15.0" peft accelerate bitsandbytes

### Check the GPU

Colab's free tier gives a Tesla T4 (about 15 GB). The T4 has no bfloat16 support, which is why the training arguments below set fp16. On my v1 run (2026-11-27), Unsloth reported that fp16 does not work for Gemma 4 and ran it in float32 instead.

In [ ]:
!nvidia-smi

## 2. Load the model and add LoRA adapters

The base model is `unsloth/gemma-4-E2B-it-unsloth-bnb-4bit`, Gemma 4 E2B instruction tuned and quantized to 4 bits so it fits comfortably on a T4.

LoRA trains small adapter matrices instead of all the model weights. I use rank 16, alpha 16 and no dropout, applied to the attention (`q/k/v/o_proj`) and MLP (`gate/up/down_proj`) layers. `use_gradient_checkpointing="unsloth"` trades a little speed for much lower memory use.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("Gemma 4 E2B loaded with LoRA adapters")

## 3. Synthetic training data

The agent is meant to call Java methods. There are three tools, plus a `none` answer for requests it should not act on:

| Tool | Parameters |
|---|---|
| `chargeLine` | `lineId`, `amount` |
| `checkLineStatus` | `lineId` |
| `emergencyStop` | `lineId` |
| `none` | `reason`: `missing_line_id`, `missing_amount` or `unsupported_request` |

The cell below generates 240 examples with `random.seed(3407)`: 80 `chargeLine`, 60 `checkLineStatus`, 60 `emergencyStop` and 40 `none`. Each class has several phrasing templates. Line ids are random between 1 and 99, and amounts come from a fixed list between 5 and 200. Some charge templates add a currency word (bucks, euros, units). The `none` examples cover an amount with no line id ("Top up 50"), a line id with no amount ("Charge line 12"), and requests that have nothing to do with the tools ("Write me a poem").

Each target is one short thought line, then the JSON action:

```
<|think|>
Action: chargeLine. ID: 12. Amt: 50.
<|channel|>{"tool": "chargeLine", "parameters": {"lineId": "12", "amount": 50.0}}
```

```
<|think|>
Action: none. Reason: missing_line_id.
<|channel|>{"tool": "none", "parameters": {"reason": "missing_line_id"}}
```

The system prompt lists the three tools and `none` with their parameters. It never contains the answer for a specific request. The reason codes are left out of the system prompt to keep every example under the 128 token limit, so the model learns them from the training data.

The prompt tokens (`<|turn|>`, `<|think|>`, `<|channel|>`) are written by hand here, so they may not match Gemma's official chat template exactly. Training and inference use the same hand-written format, which is what matters for this experiment.

In [ ]:
import json
import random
from collections import Counter
from datasets import Dataset

random.seed(3407)

SYSTEM_PROMPT = (
    "Agent. Tools: chargeLine(lineId, amount), checkLineStatus(lineId), "
    "emergencyStop(lineId), none(reason)."
)

AMOUNTS = [5, 10, 15, 20, 25, 30, 40, 50, 60, 75, 100, 150, 200]
CURRENCIES = ["bucks", "euros", "units"]

CHARGE_TEMPLATES = [
    "Charge {amt} to line {id}",
    "Top up line {id} with {amt}",
    "Add {amt} credit to line {id}",
    "Put {amt} on line {id}",
    "Line {id} needs {amt} more",
    "Charge {amt} {cur} to line {id}",
    "Top up line {id} with {amt} {cur}",
    "Put {amt} {cur} on line {id}",
    "Line {id} needs {amt} {cur} more",
]
STATUS_TEMPLATES = [
    "Is line {id} active?",
    "Check status of line {id}",
    "What's going on with line {id}",
    "Status for line {id} please",
    "Is {id} working?",
]
STOP_TEMPLATES = [
    "Stop line {id} now",
    "Emergency stop on line {id}",
    "Kill line {id} immediately",
    "Shut down line {id}, it is compromised",
    "Cut line {id} right away",
]
MISSING_ID_TEMPLATES = [
    "Top up {amt}",
    "Charge {amt} please",
    "Add {amt} {cur} of credit",
    "Put {amt} on my line",
]
MISSING_AMOUNT_TEMPLATES = [
    "Charge line {id}",
    "Top up line {id}",
    "Add credit to line {id}",
    "Put some money on line {id}",
]
UNSUPPORTED_PROMPTS = [
    "What's the weather in Amman?",
    "Write me a poem",
    "Tell me a joke",
    "Who won the football match yesterday?",
    "Translate good morning into Spanish",
    "What time is it in Tokyo?",
    "Recommend a good movie",
    "How do I bake bread?",
    "Explain how a black hole works",
    "Help me write a cover letter",
    "What's 17 times 23?",
    "Order me a pizza",
]


def fill(template):
    return template.format(id=random.randint(1, 99),
                           amt=random.choice(AMOUNTS),
                           cur=random.choice(CURRENCIES))


def charge_example():
    line_id = str(random.randint(1, 99))
    amount = random.choice(AMOUNTS)
    template = random.choice(CHARGE_TEMPLATES)
    return {
        "content": template.format(id=line_id, amt=amount, cur=random.choice(CURRENCIES)),
        "thought": f"Action: chargeLine. ID: {line_id}. Amt: {amount}.",
        "action": {"tool": "chargeLine",
                   "parameters": {"lineId": line_id, "amount": float(amount)}},
    }


def line_example(tool, templates):
    line_id = str(random.randint(1, 99))
    return {
        "content": random.choice(templates).format(id=line_id),
        "thought": f"Action: {tool}. ID: {line_id}.",
        "action": {"tool": tool, "parameters": {"lineId": line_id}},
    }


def none_example(content, reason):
    return {
        "content": content,
        "thought": f"Action: none. Reason: {reason}.",
        "action": {"tool": "none", "parameters": {"reason": reason}},
    }


dataset_samples = []
dataset_samples += [charge_example() for _ in range(80)]
dataset_samples += [line_example("checkLineStatus", STATUS_TEMPLATES) for _ in range(60)]
dataset_samples += [line_example("emergencyStop", STOP_TEMPLATES) for _ in range(60)]
dataset_samples += [none_example(fill(random.choice(MISSING_ID_TEMPLATES)), "missing_line_id")
                    for _ in range(14)]
dataset_samples += [none_example(fill(random.choice(MISSING_AMOUNT_TEMPLATES)), "missing_amount")
                    for _ in range(14)]
dataset_samples += [none_example(p, "unsupported_request") for p in UNSUPPORTED_PROMPTS]
random.shuffle(dataset_samples)

assert len(dataset_samples) == 240
print(Counter(s["action"]["tool"] for s in dataset_samples))


def format_chat(samples):
    return [{"text": (
        f"<bos><|turn|>system\n{SYSTEM_PROMPT}<|turn|>user\n{s['content']}"
        f"<|turn|>assistant\n<|think|>\n{s['thought']}\n"
        f"<|channel|>{json.dumps(s['action'])}<eos>"
    )} for s in samples]


dataset = Dataset.from_list(format_chat(dataset_samples))
print(dataset[0]["text"])

## 4. Tokenize and train

Examples are padded or truncated to 128 tokens, with the pad token set to the end-of-sequence token. Before padding, the cell checks that no example is longer than 128 tokens, so nothing gets cut off silently. The text already starts with `<bos>`, so I tokenize with `add_special_tokens=False`, which matches inference. Unsloth wraps the tokenizer, so `tokenizer_only` pulls out the underlying Hugging Face tokenizer.

I train with the plain Hugging Face `Trainer` because the Unsloth chat-template route failed for me on this setup.

| Setting | Value |
|---|---|
| Batch size | 1, with 8 gradient accumulation steps (effective batch of 8) |
| Steps | 90 (about 3 passes over 240 examples) |
| Learning rate | 2e-4 |
| Optimizer | `adamw_8bit` |
| Precision | `fp16=True` in the arguments, though on my v1 run Unsloth ran Gemma 4 in float32 |

The loss is logged every step.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

tokenizer_only = tokenizer.tokenizer if hasattr(tokenizer, "tokenizer") else tokenizer
tokenizer_only.pad_token = tokenizer_only.eos_token

# padding and truncating to 128 would silently cut off long examples, so check first
# the text already starts with <bos>, so add no special tokens (same as inference)
lengths = [len(tokenizer_only(t, add_special_tokens=False)["input_ids"])
           for t in dataset["text"]]
print(f"Longest example: {max(lengths)} tokens")
assert max(lengths) <= 128, "an example is longer than max_length and would be truncated"

def tokenize(element):
    return tokenizer_only(element["text"], truncation=True,
                          max_length=128, padding="max_length",
                          add_special_tokens=False)

tokenized_dataset = dataset.map(tokenize, batched=True,
                                remove_columns=dataset.column_names)

In [ ]:
trainer = Trainer(
    model = model,
    train_dataset = tokenized_dataset,
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer_only, mlm=False),
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        max_steps = 90,
        learning_rate = 2e-4,
        fp16 = True,
        optim = "adamw_8bit",
        gradient_checkpointing = True,
        gradient_checkpointing_kwargs = {"use_reentrant": False},
        output_dir = "tool_call_agent_v2",
        report_to = "none",
        logging_steps = 1,
    ),
)

trainer.train()

## 5. Inference

In v1 I went through several fixes to get clean JSON out of this model: greedy decoding with a stop string, priming the answer with `Action:`, a raw `USER: ... AGENT_ACTION:` format, pre-filling up to the opening `{`, and a format hint in the system prompt. The final v1 version pre-filled `Action: chargeLine.` and the opening brace, so the model never chose the tool.

This version pre-fills only up to `<|think|>\nAction:`. The model writes the tool name, the rest of the thought line, the `<|channel|>` token and the JSON itself. Decoding is greedy with at most 80 new tokens, and generation stops at the end-of-sequence token.

To get the JSON out, I decode with special tokens kept, take the text after the first `<|channel|>`, and pull out the first balanced `{...}` object by counting braces. If there isn't one, `tool_call_agent` returns `None`.

In [ ]:
FastLanguageModel.for_inference(model)

def generate_raw(user_input):
    # pre-fill only up to "Action:" so the model writes the tool name itself
    prompt = (
        f"<bos><|turn|>system\n{SYSTEM_PROMPT}<|turn|>user\n{user_input}"
        f"<|turn|>assistant\n<|think|>\nAction:"
    )
    inputs = tokenizer_only(prompt, return_tensors="pt",
                            add_special_tokens=False).to("cuda")
    outputs = model.generate(
        input_ids = inputs.input_ids,
        attention_mask = inputs.attention_mask,
        max_new_tokens = 80,
        do_sample = False,          # greedy decoding
        eos_token_id = tokenizer_only.eos_token_id,
    )
    new_tokens = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer_only.decode(new_tokens, skip_special_tokens=False)

def extract_json(text):
    # take the text after the first <|channel|> and return the first balanced {...}
    marker = "<|channel|>"
    if marker not in text:
        return None
    text = text.split(marker, 1)[1]
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]
    return None

def tool_call_agent(user_input):
    return extract_json(generate_raw(user_input))

### Try it

The prompts below cover each tool and one request the agent should decline. Edit them to test your own.

In [ ]:
test_prompts = [
    "Top up line 9 with 50 bucks.",
    "Is line 31 up and running?",
    "Shut off line 44, someone stole the phone.",
    "Add 20 euros please.",
]

for p in test_prompts:
    print(f"INPUT:  {p}")
    print(f"OUTPUT: {tool_call_agent(p)}\n")

### Parse the output

The backend needs valid JSON with the right fields, so `parse_action` checks every output against a schema table before anything runs:

- the tool must be `chargeLine`, `checkLineStatus`, `emergencyStop` or `none`
- the parameter keys must match that tool exactly
- `lineId` must be a string of digits
- `amount` must be a positive number
- `none` needs one of the three reason codes

Anything else returns `None` and should be rejected by the caller.

In [ ]:
import math
import re

TOOL_SCHEMAS = {
    "chargeLine": {"lineId", "amount"},
    "checkLineStatus": {"lineId"},
    "emergencyStop": {"lineId"},
    "none": {"reason"},
}
NONE_REASONS = {"missing_line_id", "missing_amount", "unsupported_request"}

def parse_action(raw):
    if raw is None:
        return None
    try:
        action = json.loads(raw)
    except json.JSONDecodeError:
        return None
    if not isinstance(action, dict) or set(action) != {"tool", "parameters"}:
        return None
    tool, params = action["tool"], action["parameters"]
    if tool not in TOOL_SCHEMAS or not isinstance(params, dict):
        return None
    if set(params) != TOOL_SCHEMAS[tool]:
        return None
    if "lineId" in params:
        line_id = params["lineId"]
        if not isinstance(line_id, str) or not re.fullmatch(r"[0-9]+", line_id):
            return None
    if "amount" in params:
        amount = params["amount"]
        if isinstance(amount, bool) or not isinstance(amount, (int, float)):
            return None
        if not math.isfinite(amount) or amount <= 0:
            return None
    if tool == "none" and params["reason"] not in NONE_REASONS:
        return None
    return action

for p in test_prompts:
    print(p, "->", parse_action(tool_call_agent(p)))

## 6. Evaluation

`data/test_prompts.jsonl` holds 60 hand-written requests with the expected JSON for each: 20 `chargeLine`, 15 `checkLineStatus`, 15 `emergencyStop` and 10 `none` (3 `missing_line_id`, 3 `missing_amount`, 4 `unsupported_request`). I wrote them with different verbs, word order and context from the training templates, and some amounts (35, 45, 80, 90, 120, 250, 500) never appear in training. One cell asserts that no test prompt appears word for word in the training prompts.

For each prompt the cells below generate an output, extract the JSON, run `parse_action`, and report:

- valid JSON: the extracted object parses as JSON
- tool accuracy: `parse_action` accepts the output and the tool matches the expected one
- exact match: the tool and every parameter match the expected object

Then they show a per-tool table (n, tool correct, exact) and every failure with the prompt, the expected JSON and what the model wrote. When no JSON could be extracted, the output column shows the raw generation.

The first cell clones the repo if the test file isn't next to the notebook, which is the usual case in Colab. `BRANCH` sets which branch it clones.

In [ ]:
import os
import subprocess

BRANCH = "v2"
REPO_URL = "https://github.com/mohammadhamayel23/tool-call-agent.git"

TEST_FILE = "data/test_prompts.jsonl"
if not os.path.exists(TEST_FILE):
    clone_dir = f"tool-call-agent-{BRANCH}"
    if not os.path.exists(clone_dir):
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        REPO_URL, clone_dir], check=True)
    TEST_FILE = f"{clone_dir}/data/test_prompts.jsonl"

with open(TEST_FILE, encoding="utf-8") as f:
    test_cases = [json.loads(line) for line in f if line.strip()]
print(f"Loaded {len(test_cases)} test cases from {TEST_FILE}")

In [ ]:
train_prompts = {s["content"].strip() for s in dataset_samples}
overlap = [c["prompt"] for c in test_cases if c["prompt"].strip() in train_prompts]
assert not overlap, f"test prompts found in training data: {overlap}"
print("No test prompt appears verbatim in the training prompts")

In [ ]:
import pandas as pd

rows = []
for case in test_cases:
    raw_text = generate_raw(case["prompt"])
    extracted = extract_json(raw_text)
    try:
        json.loads(extracted)
        valid_json = True
    except (TypeError, json.JSONDecodeError):
        valid_json = False
    action = parse_action(extracted)
    expected = case["expected"]
    tool_correct = action is not None and action["tool"] == expected["tool"]
    exact = action is not None and action == expected
    rows.append({
        "prompt": case["prompt"],
        "expected_tool": expected["tool"],
        "expected": json.dumps(expected),
        "output": extracted if extracted is not None else raw_text,
        "valid_json": valid_json,
        "tool_correct": tool_correct,
        "exact": exact,
    })

results = pd.DataFrame(rows)
n = len(results)
print(f"Valid JSON:    {results['valid_json'].sum()}/{n}")
print(f"Tool accuracy: {results['tool_correct'].sum()}/{n}")
print(f"Exact match:   {results['exact'].sum()}/{n} (tool and all parameters)")

In [ ]:
per_tool = (results.groupby("expected_tool")
            .agg(n=("prompt", "size"),
                 tool_correct=("tool_correct", "sum"),
                 exact=("exact", "sum"))
            .reindex(["chargeLine", "checkLineStatus", "emergencyStop", "none"]))
per_tool

In [ ]:
failures = results[~results["exact"]]
print(f"{len(failures)} failures")
pd.set_option("display.max_colwidth", None)
failures[["prompt", "expected", "output"]]

## What's next

- Add harder test cases: two requests in one message, line ids written as words, amounts with decimals.
- Use constrained decoding against the JSON schema so malformed output can't happen.
- Have the agent ask a follow-up question when the line id or amount is missing.
- Save the LoRA adapter (`model.save_pretrained(...)`) or export a merged model, and call it from a mock Java service.